# Tensor Operations — Labelled Einsum for Multivectors

**Part II · Geometric Algebra** — Tutorial 13

This tutorial introduces the tensor layer: `MVTensor` (an N-D array with
`BladeMask`-labelled axes) and `MVLabeledTensor` (label-driven arithmetic). Labels
let you express contractions, broadcasting, and transposes without writing raw
`np.einsum` subscripts.

By the end you will be able to:

- Build `MVTensor`s with factories and convert to/from multivectors.
- Build the **product tensor** with `product_tensor()`.
- Compute a GA product by label contraction: `O["kij"] * A["i"] * B["j"]`.
- Distinguish contraction (`*`) from element-wise (`_`) labels.
- Broadcast with `+` / `-` and reorder axes with arrow syntax (`"ij->ji"`).
- Iterate per-element with `iter_labels`.

> **Prerequisites:** [Tutorial 10](../10_blade_mask/) (blade masks) and
> [Tutorial 12](../12_matrix/) (matrix primitives).

## 1. Setup

The tensor types live in `pytanga.tensor`, with helpers in
`pytanga.tensor.convert`, `.ops`, and `.product`. `iter_labels` lives in
`pytanga.tensor._labeled`.

In [1]:
from pytanga import BladeMask
from pytanga.basis import BasisE3
from pytanga.tensor import MVLabeledTensor, MVTensor
from pytanga.tensor._labeled import iter_labels
from pytanga.tensor.convert import from_tensor, to_tensor
from pytanga.tensor.ops import contract
from pytanga.tensor.product import product_tensor

E3 = BasisE3()
full = BladeMask.full(E3)     # all 8 blades

## 2. `MVTensor` — an N-D array with labelled axes

`MVTensor.zeros([spec, ...])` builds a zero tensor: a `BladeMask` spec creates a
blade axis of that size, and a plain `int` spec creates an unlabelled batch axis
(`mask=None`).

In [2]:
t = MVTensor.zeros([full])           # one multivector -> (8,)
batch = MVTensor.zeros([5, full])    # batch of 5 -> (5, 8), axis 0 is mask=None

print("t      :", t.shape, " masks:", [None if m is None else "blade" for m in t.masks])
print("batch  :", batch.shape, " masks:", [None if m is None else "blade" for m in batch.masks])
print("algebra:", type(t.algebra).__name__)
print("scaled :", t.mul_scalar(2.0).shape)

t      : (8,)  masks: ['blade']
batch  : (5, 8)  masks: [None, 'blade']
algebra: BasisE3
scaled : (8,)


## 3. `to_tensor` / `from_tensor`

`to_tensor(mv, mask=...)` fills a rank-1 tensor from a multivector; a list of MVs
becomes a rank-2 tensor with a trailing batch axis. `from_tensor` reverses the
conversion.

In [3]:
a = to_tensor(E3("2 e1 - 3 e2"), mask=full)
print("single shape :", a.shape)
print("from_tensor  :", from_tensor(a).to_dict())

batch = to_tensor([E3("e1"), E3("e2")], mask=full)
print("batch shape  :", batch.shape, " masks:", [None if m is None else "blade" for m in batch.masks])
print("from batch   :", [x.to_dict() for x in from_tensor(batch)])

single shape : (8,)
from_tensor  : {'s': 0.0, 'e1': 2.0, 'e2': -3.0, 'e12': 0.0, 'e3': 0.0, 'e13': 0.0, 'e23': 0.0, 'I': 0.0}
batch shape  : (8, 2)  masks: ['blade', None]
from batch   : [{'e1': 1.0}, {'e2': 1.0}]


## 4. `product_tensor` — the bilinear product as a 3-D tensor

`product_tensor(a_mask, b_mask)` returns an `MVTensor` with masks
`(c_mask, a_mask, b_mask)`: axis 0 is the result blade, axis 1 the left operand,
axis 2 the right operand. Its entries are `-1`, `0`, or `1`.

In [4]:
O = product_tensor(full, full)     # geometric product tensor

print("shape  :", O.shape)                 # (8, 8, 8)
print("masks  :", [len(m) for m in O.masks])
print("values :", int(O.data.min()), "..", int(O.data.max()))

shape  : (8, 8, 8)
masks  : [8, 8, 8]
values : -1 .. 1


## 5. Label-driven contraction

Indexing an `MVTensor` with a string creates an `MVLabeledTensor`. Multiplying two
labeled tensors contracts shared `*`-labels automatically, so the geometric product
is just `O["kij"] * A["i"] * B["j"]`.

In [5]:
a = to_tensor(E3("e1"), mask=full)     # A_i
b = to_tensor(E3("e2"), mask=full)     # B_j

c = O["kij"] * a["i"] * b["j"]         # labels "k*"
print("labels :", c.labels)
print("GP     :", from_tensor(c.tensor).prune().to_dict())
print("direct :", (E3("e1") * E3("e2")).to_dict())

labels : k*
GP     : {'e12': 1.0}
direct : {'e12': 1.0}


## 6. Label modes, broadcasting, and transposes

- A shared label with mode `*` is **contracted** (summed); mode `_` is
  **element-wise** (kept).
- `+` / `-` broadcast over non-matching labels (the output is the union of labels).
- Arrow syntax (`"ij->ji"`) reorders axes.

In [6]:
# Element-wise batch axis: 'n' has mode '_', so it is NOT summed.
P = MVLabeledTensor.zeros("i*n_", [full, 5])
Q = MVLabeledTensor.zeros("j*n_", [full, 5])
R = P["in_"] * Q["jn_"]
print("element-wise product:", R.labels, R.shape)     # i*j*n_  (8, 8, 5)

# '+' broadcasts over non-matching labels.
A = MVLabeledTensor.zeros("i", [full])
B = MVLabeledTensor.zeros("j", [full])
print("A['i'] + B['j']     :", (A["i"] + B["j"]).labels, (A["i"] + B["j"]).shape)

# Arrow syntax transposes axes.
T = MVLabeledTensor.zeros("ij", [full, BladeMask(E3, [0, 1, 2])])
print("T['ij']             :", T.labels, T.shape)
print("T['ij->ji']         :", T["ij->ji"].labels, T["ij->ji"].shape)

element-wise product: i*j*n_ (8, 8, 5)
A['i'] + B['j']     : i*j* (8, 8)
T['ij']             : i*j* (8, 3)
T['ij->ji']         : j*i* (3, 8)


## 7. `iter_labels` — per-element computation

`iter_labels("n", ...)` iterates over a named batch axis, yielding slices with that
axis **removed**. Use it for per-element work that is not a single einsum.

In [7]:
A_batch = MVLabeledTensor.zeros("n*a*", [4, full])
B_batch = MVLabeledTensor.zeros("n*b*", [4, full])

for i, (a_i, b_i) in enumerate(iter_labels("n", A_batch, B_batch)):
    gp = O["kij"] * a_i["i"] * b_i["j"]      # per-element geometric product
    print(f"element {i}: {a_i.labels} / {b_i.labels} -> {gp.labels} {gp.shape}")

element 0: a* / b* -> k* (8,)
element 1: a* / b* -> k* (8,)
element 2: a* / b* -> k* (8,)
element 3: a* / b* -> k* (8,)


## 8. Summary & next steps

| Task | API |
|---|---|
| Blade-labelled N-D array | `MVTensor.zeros([mask, 5, mask])` |
| MV ↔ tensor | `to_tensor(mv, mask=...)`, `from_tensor(t)` |
| Product tensor | `product_tensor(a_mask, b_mask)` |
| Label contraction | `O["kij"] * A["i"] * B["j"]` |
| Element-wise label | `"n_"` suffix |
| Broadcast add / sub | `A["i"] + B["j"]` |
| Transpose | `T["ij->ji"]` |
| Per-element loop | `iter_labels("n", A, B)` |

**Where to go next:**

- [**14 · Expression System**](../14_expression/) — symbolic variables and
  equations on top of the algebra.
- [**12 · Matrix Operations**](../12_matrix/) — the matrix-level view of the same
  products.